# Aula 12 · matplotlib e KPIs

Esta aula apresenta o [capítulo 12 do site](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/). A ideia central: **todo gráfico responde a uma
pergunta**. Escreva a pergunta antes, e ela escolhe o gráfico: **linha** para como algo
evoluiu, **barras** para comparar, **histograma** para ver como os valores se
distribuem.

**Ao fim da aula você consegue:**

1. desenhar linha, barras e histograma com título, eixos com unidade e legenda;
2. acrescentar a linha de referência (limite, SLA) que dá sentido ao gráfico;
3. reconhecer quando a escala de um gráfico engana.

**Roteiro:** 🔥 aquecimento · 📟 chamado · 1. linha · 2. barras · 3. o gráfico que
engana · 4. histograma · 5. várias séries · 📟 resolvendo o chamado · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

A célula ⚙️ desta aula também **baixa o mês de medições** (`medicoes_mes.csv`) do
site do curso, o mesmo da Aula 11.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
import os
import urllib.request

ARQUIVO = "medicoes_mes.csv"
if not os.path.exists(ARQUIVO):
    urllib.request.urlretrieve(
        "https://lacouth.github.io/python_telecom-site/dados/medicoes_mes.csv", ARQUIVO)
print(ARQUIVO, "pronto")

## 🔥 Aquecimento — da aula passada

Sem rodar nada: o que cada linha responde, em português?

```python
df[df["estado"] == "DOWN"].groupby("enlace").size()
(df.groupby("enlace")["no_ar"].mean() * 100).round(2)
df.sort_values("trafego_mbps", ascending=False).head(3)
```

<details>
<summary><b>Resposta</b></summary>

1. Quantas horas cada enlace ficou fora do ar (filtra, agrupa, conta).
2. A disponibilidade de cada enlace, em % — a média de uma coluna `True`/`False` é a
   fração de `True`.
3. As três horas de maior tráfego do mês, com a linha inteira de cada uma.

Hoje, esses números viram figuras.

</details>

## 📟 O chamado de hoje

> **Chamado #1215 — NOC Maré Net**
>
> *"Estagiário, a reunião mensal com a diretoria é amanhã. Eles não leem tabela. Preciso
> de **uma figura** que mostre, de relance, quais OLTs descumpriram o SLA de 99,5% em
> março — e de outra que explique por que a equipe de campo tem que visitar a
> OLT-SUL-03 esta semana, mesmo ela nunca tendo caído."*

No fim da aula você monta as duas figuras.

In [ ]:
# 📦 dados prontos — só rode esta célula
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("medicoes_mes.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["dia"] = df["timestamp"].dt.day
df["hora"] = df["timestamp"].dt.hour
df["no_ar"] = df["estado"] == "UP"

## 1. Linha: como algo evoluiu

**Pergunta:** a potência da OLT-SUL-03 está caindo? Quando passou do alerta?

As peças: `plt.figure` abre a figura, `plt.plot(x, y)` desenha, `plt.axhline` traça a
referência, `plt.title`/`xlabel`/`ylabel` dão nome a tudo (**com unidade**) e
`plt.show()` mostra.

📖 [capítulo 12 · Linha: como algo evoluiu](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#linha-como-algo-evoluiu)

**✍️ Passo 1.** Com a tabela da célula de dados, calcule
`por_dia = df[df["enlace"] == "OLT-SUL-03"].groupby("dia")["potencia_dbm"].mean()`.
Depois: `plt.figure(figsize=(8, 4))`, `plt.plot(por_dia.index, por_dia.values, marker="o")`
e `plt.show()`.

In [ ]:
# ✍️ passo 1

**Preveja:** que formato a linha vai ter?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Uma descida quase reta, de −22 para −26,5 dBm ao longo do mês. O gráfico já mostra a
tendência — mas quem não conhece o assunto não sabe se −25 é bom ou ruim, nem o que
é o eixo.

</details>

**✍️ Passo 2.** Repita o gráfico acrescentando, antes do `show`: `plt.axhline(-25, color="orange",
linestyle="--", label="alerta")`, um título, `plt.xlabel("dia do mês")`,
`plt.ylabel("potência recebida (dBm)")` e `plt.legend()`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que a linha tracejada acrescenta para quem lê?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A **referência**: agora dá para ver, sem saber nada de dBm, que a linha cruza o
alerta por volta do dia 22 e continua descendo. Um gráfico sem a referência obriga
quem lê a lembrar qual é o valor ruim.

</details>

### 🎯 Sua vez — O dia do alerta

Escreva `dia_do_alerta(df, enlace, limite)`, que devolve o **primeiro dia** em que a
potência média do enlace ficou abaixo do `limite` (um `int`), ou `None` se isso nunca
aconteceu.

In [ ]:
def dia_do_alerta(df, enlace, limite):
    # sua solução aqui
    pass

In [ ]:
confere(dia_do_alerta, [
    ((df, "OLT-SUL-03", -25), 22),
    ((df, "OLT-SUL-03", -23), 8),
    ((df, "OLT-CENTRO-01", -25), None),
])

<details>
<summary><b>💡 Dica</b></summary>

A média por dia é a do passo. Filtre os dias com média abaixo do limite
(`por_dia[por_dia < limite]`); se sobrar algum (`len(...) > 0`), o primeiro está em
`.index[0]` — converta com `int(...)`.

</details>

## 2. Barras: comparar grupos

**Pergunta:** quais enlaces descumpriram o SLA de 99,5%?

📖 [capítulo 12 · Barras: comparar grupos](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#barras-comparar-grupos)

**✍️ Passo 3.** Calcule `disp = df.groupby("enlace")["no_ar"].mean() * 100`. Faça
`plt.figure(figsize=(8, 4))`, `plt.bar(disp.index, disp.values)`, a linha do SLA com
`plt.axhline(99.5, color="red", linestyle="--", label="SLA")`, o `ylabel` com `%`, a
legenda e o `show`.

In [ ]:
# ✍️ passo 3

**Preveja:** dá para ver quem ficou abaixo do SLA?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Quase não dá: as cinco barras parecem do mesmo tamanho, todas perto de 100, e a linha
do SLA fica colada no topo. O eixo começa no **zero**, e diferenças de menos de 1% ficam
invisíveis nessa escala. A próxima seção trata disso.

</details>

**✍️ Passo 4.** Repita o gráfico com `plt.ylim(98, 100.2)` antes do `show`.

In [ ]:
# ✍️ passo 4

**Preveja:** e agora?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Agora a OLT-LESTE-04 e a OLT-OESTE-05 aparecem claramente abaixo da linha vermelha. O
`ylim` escolheu a faixa do eixo vertical — e isso tem consequência.

</details>

## 3. O gráfico que engana

O mesmo dado, em duas escalas, dá impressões opostas: com o eixo a partir de zero,
tudo parece igual; com o eixo cortado, uma diferença pequena pode parecer enorme.

📖 [capítulo 12 · O gráfico que engana](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#o-grafico-que-engana)

> ⚠️ **Armadilha.** Recortar o eixo de um gráfico de barras sem dizer. Em disponibilidade, o recorte
> mostra o que importa (99,7% contra 99,2% é **o triplo** de horas fora do ar); mas o
> mesmo recurso também serve para fazer uma diferença irrelevante parecer grave. A
> regra: **em barras, o eixo começa no zero — a não ser que você tenha um motivo e o
> diga** ("eixo de 98% a 100%", ao lado do gráfico).

### 🎯 Sua vez — Quantas horas isso dá?

Para dizer a diferença em horas, em vez de %, escreva
`horas_fora(disponibilidade, horas=744)`, que converte uma disponibilidade em % no
número de horas fora do ar no período, com 1 casa.

In [ ]:
def horas_fora(disponibilidade, horas=744):
    # sua solução aqui
    pass

In [ ]:
confere(horas_fora, [
    ((99.731,), 2.0),
    ((99.194,), 6.0),
    ((99.5,), 3.7),
    ((100,), 0.0),
])

<details>
<summary><b>💡 Dica</b></summary>

A fração fora do ar é `(100 - disponibilidade) / 100`; vezes as horas do período.

</details>

## 4. Histograma: como os valores se distribuem

**Pergunta:** as medições de potência estão concentradas numa faixa segura?

`plt.hist(valores, bins=30)` divide a faixa em 30 intervalos e desenha quantas medições
caíram em cada um.

📖 [capítulo 12 · Histograma: como os valores se distribuem](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#histograma-como-os-valores-se-distribuem)

**✍️ Passo 5.** Faça `plt.figure(figsize=(8, 4))`, `plt.hist(df["potencia_dbm"].dropna(), bins=30)`, a
linha da sensibilidade com `plt.axvline(-27, color="red", linestyle="--")`, os dois
eixos com nome e o `show`.

In [ ]:
# ✍️ passo 5

**Preveja:** por que o `.dropna()`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

As horas fora do ar não têm potência (`NaN`); o `dropna()` as tira antes de desenhar.
O histograma mostra picos altos — cada enlace estável em volta da sua potência — e uma
faixa baixa e espalhada de −22 a −27 dBm: a OLT-SUL-03, que passou um pouco de tempo
em cada valor enquanto descia.

</details>

### 🎯 Sua vez — Quantas medições na faixa

O histograma conta medições por faixa. Escreva `quantas_na_faixa(df, minimo, maximo)`,
que devolve quantas medições de potência ficaram **de `minimo` (inclusive) até
`maximo` (exclusive)**, como `int`.

In [ ]:
def quantas_na_faixa(df, minimo, maximo):
    # sua solução aqui
    pass

In [ ]:
confere(quantas_na_faixa, [
    ((df, -27, -25), 244),
    ((df, -25, -20), 2722),
    ((df, -20, -19), 738),
])

<details>
<summary><b>💡 Dica</b></summary>

Uma máscara com duas condições — `>= minimo` e `< maximo` —, cada uma entre
parênteses, ligadas por `&`; some e converta com `int(...)`.

</details>

## 5. Várias séries no mesmo gráfico

**Pergunta:** o horário de pico é o mesmo em todo enlace? Chamar `plt.plot` várias
vezes, antes do `show`, desenha várias linhas na mesma figura — cada uma com o seu
`label`.

📖 [capítulo 12 · Várias séries no mesmo gráfico](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#varias-series-no-mesmo-grafico)

**✍️ Passo 6.** Abra a figura e, num `for enlace in ["OLT-CENTRO-01", "OLT-LESTE-04"]:`, calcule a
média do tráfego por hora daquele enlace e faça
`plt.plot(por_hora.index, por_hora.values, marker=".", label=enlace)`. Depois do laço:
título, `xlabel("hora do dia")`, `ylabel("tráfego (Mbit/s)")`, legenda e `show`.

In [ ]:
# ✍️ passo 6

**Preveja:** as duas curvas vão ter o mesmo formato, se os volumes são tão diferentes?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Volumes bem diferentes — a OLT-CENTRO-01 tem o dobro do tráfego —, mas o **formato** é o
mesmo: vale às 4h, pico às 21h. É o perfil de uma rede residencial. A legenda é o que
permite dizer qual curva é qual.

</details>

## 📟 Resolvendo o chamado

A primeira figura do chamado: as barras de disponibilidade com o SLA.

### 🎯 Sua vez — A figura do SLA

Escreva `grafico_sla(df, sla)`, que **desenha** as barras da disponibilidade de cada
enlace com a linha do SLA, eixo y de 98 a 100,2 (e diga isso no título), eixo y com
`%` — e **devolve** a lista ordenada dos enlaces abaixo do SLA. Chame `plt.show()` no
fim da função.

In [ ]:
def grafico_sla(df, sla):
    # sua solução aqui
    pass

In [ ]:
confere(grafico_sla, [((df, 99.5), ["OLT-LESTE-04", "OLT-OESTE-05"])])

<details>
<summary><b>💡 Dica</b></summary>

A disponibilidade é a do passo da seção 2. Desenhe como lá, com `plt.ylim(98, 100.2)`;
antes do `return`, o filtro `disp[disp < sla]` e `sorted(....index.tolist())`.

</details>

A segunda figura do chamado é a linha da seção 1: a potência da OLT-SUL-03 cruzando o
alerta no dia 22 e seguindo para a sensibilidade da ONU no começo de abril. Ela nunca
caiu — e é exatamente por isso que o gráfico importa: nenhum alarme de "fora do ar"
teria avisado.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** Para mostrar a **evolução** do tráfego ao longo do mês, o gráfico é:
a) barras  b) linha  c) histograma  d) pizza

<details>
<summary><b>Resposta da 1</b></summary>

**b**. Linha para evolução no tempo; barras para comparar grupos; histograma para
distribuição.

</details>

**2.** O arquivo salvo com `savefig` saiu em branco. A causa mais provável é:
a) faltou `plt.figure()`  b) o `savefig` veio depois do `plt.show()`
c) faltou `dpi`  d) o nome do arquivo não tem extensão

<details>
<summary><b>Resposta da 2</b></summary>

**b**. Depois de mostrada, a figura é descartada: salve antes de mostrar.

</details>

**3.** Duas barras de 99,7% e 99,2% parecem iguais num gráfico. O que está
acontecendo?
a) o matplotlib arredondou  b) o eixo começa no zero e a diferença é pequena nessa
escala  c) faltou a legenda  d) as barras estão sobrepostas

<details>
<summary><b>Resposta da 3</b></summary>

**b**. Recortar o eixo mostra a diferença — e o recorte precisa ser dito.

</details>

## 🏠 Para casa

- [Lista 12](https://lacouth.github.io/python_telecom-site/listas/lista12/) —
  matplotlib e KPIs, com testes automáticos no Colab.
- Releia o [capítulo 12 do site](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/), principalmente o checklist antes de pôr um gráfico num
  relatório: 📖 [capítulo 12 · Checklist de um gráfico de relatório](https://lacouth.github.io/python_telecom-site/unidade6-dados/12-matplotlib-kpis/#checklist-de-um-grafico-de-relatorio).
- **Semana que vem:** Avaliação 2 (capítulos 9 a 12). Refaça as listas 09 a 12.